# Préparation des données - UCI Consommation électrique domestique

Ce notebook fait suite à 01_exploration_uci_power.ipynb.
L'objectif est de préparer les données brutes pour alimenter l'entrepôt de données (Azure SQL Database) via le
pipeline Azure Data Factory > Databricks > SQL.

Les étapes de ce notebook sont :
1. Chargement du fichier brut nettoyé depuis data/raw/.
2. Nettoyage : suppression des lignes contenant des valeurs manquantes.
3. Agrégation horaire : une ligne par heure, KPI de consommation agrégés.
4. Agrégation journalière : une ligne par jour, KPI de consommation agrégés.
5. Export des tables préparées vers data/processed/ au format CSV et Parquet.

Ces tables préparées simuleront les sorties de la couche Databricks (curated zone) dans l'architecture Azure.

## Chargement

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path().resolve().parent
data_path = project_root / "data" / "raw" / "household_power_consumption.txt"

raw_df = pd.read_csv(
    data_path,
    sep=";",
    na_values=["?"],
    low_memory=False
)

raw_df["datetime"] = pd.to_datetime(
    raw_df["Date"] + " " + raw_df["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

raw_df = raw_df.drop(columns=["Date", "Time"]).set_index("datetime").sort_index()

raw_df.shape

(2075259, 7)

Nous rechargeons le fichier brut UCI et appliquons les mêmes transformations que dans le notebook d'exploration :
conversion de Date+Time en index datetime, suppression des colonnes texte d'origine.

Ce chargement constitue le point de départ de la chaîne de préparation, équivalent à ce que ferait
un notebook Databricks lisant depuis la zone raw du Data Lake Azure.

## Nettoyage des valeurs manquantes

In [2]:
# Nombre de lignes avant nettoyage
print("Avant nettoyage :", len(raw_df))

# Suppression des lignes avec au moins une valeur manquante
clean_df = raw_df.dropna()

# Vérification
print("Après nettoyage :", len(clean_df))
print("Lignes supprimées :", len(raw_df) - len(clean_df))
print("Pourcentage supprimé :", round((len(raw_df) - len(clean_df)) / len(raw_df) * 100, 3), "%")

Avant nettoyage : 2075259
Après nettoyage : 2049280
Lignes supprimées : 25979
Pourcentage supprimé : 1.252 %


Nous appliquons dropna() pour supprimer toutes les lignes contenant au moins une valeur manquante.

Le résultat confirme les observations faites lors de l'exploration :
- 25 979 lignes supprimées sur 2 075 259, soit 1,252 % du dataset.
- Il reste 2 049 280 lignes propres, ce qui représente plus de 98,7 % des données originales.

Ce taux de suppression est très faible et n'affecte pas la représentativité du jeu de données.
Dans le contexte de notre plateforme BI, cette stratégie de suppression simple (dropna) est justifiée pour deux raisons :
1. Le volume de données manquantes est négligeable (environ 1 ligne sur 80).
2. Les agrégations horaires et journalières que nous allons produire ensuite sont robustes à ces quelques trous, car les timestamps manquants sont répartis de manière éparse sur 47 mois.

Dans un pipeline Databricks en production, cette logique se traduirait par un filtre sur les colonnes de mesures dans une cellule Spark Python, avant d'écrire dans la zone curated du Data Lake.

## Agrégation horaire

Les données brutes sont à la minute (2 millions de lignes). Pour la plateforme BI et l'entrepôt de données,
nous avons besoin de tables agrégées à granularité plus large.

Nous allons produire deux tables :
- hourly_df : une ligne par heure, avec la somme et la moyenne de chaque mesure.
- daily_df : une ligne par jour (étape suivante).

Ces deux tables simulent les sorties de la zone curated de Databricks dans l'architecture Azure.

In [3]:
# Agrégation horaire
hourly_df = clean_df.resample("h").agg(
    Global_active_power_mean=("Global_active_power", "mean"),
    Global_active_power_sum=("Global_active_power", "sum"),
    Global_reactive_power_mean=("Global_reactive_power", "mean"),
    Voltage_mean=("Voltage", "mean"),
    Global_intensity_mean=("Global_intensity", "mean"),
    Sub_metering_1_sum=("Sub_metering_1", "sum"),
    Sub_metering_2_sum=("Sub_metering_2", "sum"),
    Sub_metering_3_sum=("Sub_metering_3", "sum"),
)

print("Shape :", hourly_df.shape)
hourly_df.head()

Shape : (34589, 8)


,Global_active_power_mean,Global_active_power_sum,Global_reactive_power_mean,Voltage_mean,Global_intensity_mean,Sub_metering_1_sum,Sub_metering_2_sum,Sub_metering_3_sum
datetime,,,,,,,,
2006-12-16 17:00:00,4.222889,152.024,0.229000,234.643889,18.100000,0.0,19.0,607.0
2006-12-16 18:00:00,3.632200,217.932,0.080033,234.580167,15.600000,0.0,403.0,1012.0
2006-12-16 19:00:00,3.400233,204.014,0.085233,233.232500,14.503333,0.0,86.0,1001.0
2006-12-16 20:00:00,3.268567,196.114,0.075100,234.071500,13.916667,0.0,0.0,1007.0
2006-12-16 21:00:00,3.056467,183.388,0.076667,237.158667,13.046667,0.0,25.0,1033.0


La méthode resample("h").agg(...) regroupe toutes les mesures minute par minute d'une même heure
en un seul enregistrement, soit 34 589 lignes contre 2 049 280 lignes brutes - une réduction par
un facteur d'environ 60, ce qui correspond exactement à 60 minutes par heure.

Choix des agrégations par colonne :
- mean pour les mesures instantanées (Global_active_power, Global_reactive_power, Voltage,
  Global_intensity) : ces variables représentent des états physiques moyens sur la période,
  la moyenne horaire est donc la statistique la plus représentative.
- sum pour les sous-compteurs (Sub_metering_1/2/3) : ces variables sont en Wh d'énergie active,
  donc additionner les valeurs à la minute donne l'énergie totale consommée sur l'heure, ce qui
  est la bonne unité de mesure pour un bilan énergétique.

Lecture des premières lignes :
Les premières lignes correspondent au début du dataset (16 décembre 2006, soirée) :
- Les puissances actives moyennes sont élevées (≈ 3 à 4 kW), ce qui est cohérent avec des heures de
  soirée où le foyer est occupé (cuisine, éclairage, appareils actifs).
- Sub_metering_1 (cuisine) est à 0 pour ces heures, tandis que Sub_metering_3 (chauffe-eau climatisation) montre des
sommes très importantes (≈ 600 à 1 000 Wh/heure), ce qui suggère
  un usage intensif du chauffe-eau ou du chauffage en soirée d'hiver.

Cette table horaire constitue la première table agrégée de notre entrepôt de données, analogue
à une table de faits à granularité horaire dans un modèle dimensionnel (star schema).

## Agrégation journalière

Nous allons maintenant produire une deuxième table agrégée à granularité journalière.
Cette table correspond à une table de faits d'instantané périodique dans un modèle dimensionnel
(star schema) : chaque ligne représente un résumé de la consommation d'un jour complet.

In [4]:
# Agrégation journalière
daily_df = clean_df.resample("D").agg(
    Global_active_power_mean=("Global_active_power", "mean"),
    Global_active_power_sum=("Global_active_power", "sum"),
    Global_reactive_power_mean=("Global_reactive_power", "mean"),
    Voltage_mean=("Voltage", "mean"),
    Global_intensity_mean=("Global_intensity", "mean"),
    Sub_metering_1_sum=("Sub_metering_1", "sum"),
    Sub_metering_2_sum=("Sub_metering_2", "sum"),
    Sub_metering_3_sum=("Sub_metering_3", "sum"),
)

print("Shape :", daily_df.shape)
daily_df.head()

Shape : (1442, 8)


,Global_active_power_mean,Global_active_power_sum,Global_reactive_power_mean,Voltage_mean,Global_intensity_mean,Sub_metering_1_sum,Sub_metering_2_sum,Sub_metering_3_sum
datetime,,,,,,,,
2006-12-16,3.053475,1209.176,0.088187,236.243763,13.082828,0.0,546.0,4926.0
2006-12-17,2.354486,3390.460,0.156949,240.087028,9.999028,2033.0,4187.0,13341.0
2006-12-18,1.530435,2203.826,0.112356,241.231694,6.421667,1063.0,2621.0,14018.0
2006-12-19,1.157079,1666.194,0.104821,241.999313,4.926389,839.0,7602.0,6197.0
2006-12-20,1.545658,2225.748,0.111804,242.308062,6.467361,0.0,2648.0,14063.0


La table daily_df contient 1 442 lignes, une par jour sur les 47 mois du dataset (décembre 2006 à
novembre 2010), contre 2 049 280 lignes brutes, soit une réduction par un facteur d'environ 1 420.

Lecture des premières lignes :

- Le 16 décembre 2006 (premier jour du dataset) montre une puissance active moyenne élevée (≈ 3,05 kW)
mais une somme relativement faible (≈ 1 209 kWh cumulées), car ce jour n'est pas complet : le
dataset commence en soirée.
- À partir du 17 décembre, les jours complets montrent des sommes journalières entre ≈ 1 600 et
≈ 3 400 kWh-minute cumulées. Ces valeurs sont cohérentes avec un foyer résidentiel français en période
hivernale, une saison de forte consommation.
- Sub_metering_3_sum (chauffe-eau/climatisation) domine largement les deux autres sous-compteurs sur
ces premiers jours d'hiver (≈ 4 000 à 14 000 Wh/jour), ce qui confirme un usage intensif du chauffage
ou du chauffe-eau.

Cette table journalière correspond à une table de faits d'instantané périodique dans notre star schema :
chaque ligne représente l'état de la consommation du foyer pour une journée complète.
Elle servira directement comme source pour les visuels Power BI (courbes de tendance, comparaisons
mensuelles, KPI d'énergie par zone) et comme base pour les calculs DAX de variation jour/jour.

## Export des tables préparées

Nous exportons les deux tables agrégées (horaire et journalière) dans le dossier data/processed/
en deux formats :

- CSV : format universel, lisible directement dans Excel, Power BI ou tout outil de reporting.
- Parquet : format colonnaire compressé, standard dans les architectures big data et
  nativement supporté par Databricks et Azure Data Lake.

Dans l'architecture Azure cible, ces fichiers simulent la sortie de la zone curated de
Databricks, prête à être chargée dans Azure SQL Database.

In [5]:
processed_path = project_root / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

print("Dossier processed prêt :", processed_path.relative_to(project_root))

Dossier processed prêt : data\processed


In [6]:
# Export de la table horaire
hourly_df.to_csv(processed_path / "hourly_power_consumption.csv")
hourly_df.to_parquet(processed_path / "hourly_power_consumption.parquet")

# Export de la table journalière
daily_df.to_csv(processed_path / "daily_power_consumption.csv")
daily_df.to_parquet(processed_path / "daily_power_consumption.parquet")

print("Export terminé. Fichiers générés :")
for f in sorted(processed_path.iterdir()):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name} - {size_kb} KB")

Export terminé. Fichiers générés :
  daily_power_consumption.csv - 159.7 KB
  daily_power_consumption.parquet - 97.2 KB
  hourly_power_consumption.csv - 3617.7 KB
  hourly_power_consumption.parquet - 1248.7 KB


## Conclusion

Ce notebook a couvert l'ensemble de la chaîne de préparation des données brutes UCI vers des
tables structurées, prêtes à alimenter l'entrepôt de données de la plateforme BI.

Ce que nous avons accompli :

1. Chargement des données brutes depuis data/raw/, avec recréation de l'index temporel.
2. Nettoyage : suppression de 25 979 lignes avec valeurs manquantes (1,252 %), conservant
plus de 98,7 % des données originales.
3. Agrégation horaire : production d'une table de 34 589 lignes résumant la consommation
heure par heure (moyennes pour les mesures instantanées, sommes pour les sous-compteurs).
4. Agrégation journalière : production d'une table de 1 442 lignes, une par jour sur 47 mois,
résumant la consommation quotidienne du foyer.
5. Export des deux tables en CSV et Parquet dans data/processed/, le format Parquet
offrant une compression d'environ 3× par rapport au CSV pour la table horaire.

Ce que ces tables représentent dans l'architecture Azure :

Ces deux tables correspondent aux sorties de la zone curated de Databricks dans notre
pipeline Azure :


data/raw/ > Azure Data Lake (zone raw)
data/processed/ > Azure Data Lake (zone curated, après transformation Databricks)


La table journalière (daily_power_consumption) constituera la base de la table de faits
de notre entrepôt Azure SQL (FactConsumption), tandis que la table horaire servira aux
analyses plus fines dans les dashboards Power BI.

Le notebook 03_schema_entrepot.ipynb abordera la conception du modèle dimensionnel
(star schema) : création des tables FactConsumption, DimDate et DimMeter, qui seront
ensuite chargées dans Azure SQL Database.